# Séparation des Données et Imputation

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

In [ ]:
df = pd.read_csv('data/raw/student_mental_health.csv')
print(f"Dimensions initiales : {df.shape[0]} lignes, {df.shape[1]} colonnes")
display(df.head(3))

Dimensions initiales : 2000 lignes, 17 colonnes


,Student_ID,Age,Gender,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Physical_Activity,Financial_Stress,Family_Support,Attendance_Rate,Part_Time_Job,Diet_Quality,Previous_Mental_Health_History,Mental_Health_Risk
0,STU0001,22,Male,3,6.6,17.6,2.91,6.9,7.5,Moderate,8.5,4.0,78.0,No,Poor,No,High
1,STU0002,21,Male,4,5.6,22.4,2.92,7.6,11.5,Low,4.5,5.3,85.4,Yes,Average,No,Medium
2,STU0003,22,Male,5,7.1,28.1,2.68,6.5,8.9,Low,4.8,6.1,79.5,No,Average,No,Medium


## Séparation des données (80% / 20%) et Imputation des Valeurs Manquantes

In [3]:
X = df.drop(columns=['Student_ID', 'Mental_Health_Risk', 'Age', 'Gender'])
y = df['Mental_Health_Risk']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("=== RÉPARTITION DES ENSEMBLES ===")
print(f"Ensemble d'entraînement (Train) : {X_train.shape[0]} exemples ({X_train.shape[0]/len(df):.0%})")
print(f"Ensemble de test (Test)         : {X_test.shape[0]} exemples ({X_test.shape[0]/len(df):.0%})")

print("\n=== VÉRIFICATION DE LA STRATIFICATION (Distribution de y) ===")
print("Train (%):\n", y_train.value_counts(normalize=True).round(3) * 100)
print("\nTest (%):\n", y_test.value_counts(normalize=True).round(3) * 100)

=== RÉPARTITION DES ENSEMBLES ===
Ensemble d'entraînement (Train) : 1600 exemples (80%)
Ensemble de test (Test)         : 400 exemples (20%)

=== VÉRIFICATION DE LA STRATIFICATION (Distribution de y) ===
Train (%):
 Mental_Health_Risk
High      34.0
Medium    33.0
Low       33.0
Name: proportion, dtype: float64

Test (%):
 Mental_Health_Risk
High      34.0
Medium    33.0
Low       33.0
Name: proportion, dtype: float64


In [4]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

print(f"Colonnes numériques ({len(num_cols)}) : {num_cols}")
print(f"Colonnes catégorielles ({len(cat_cols)}) : {cat_cols}")

num_imputer = SimpleImputer(strategy='median')

X_train_num_imputed = pd.DataFrame(
    num_imputer.fit_transform(X_train[num_cols]),
    columns=num_cols,
    index=X_train.index
)

X_test_num_imputed = pd.DataFrame(
    num_imputer.transform(X_test[num_cols]),
    columns=num_cols,
    index=X_test.index
)

cat_imputer = SimpleImputer(strategy='most_frequent')

X_train_cat_imputed = pd.DataFrame(
    cat_imputer.fit_transform(X_train[cat_cols]),
    columns=cat_cols,
    index=X_train.index
)

X_test_cat_imputed = pd.DataFrame(
    cat_imputer.transform(X_test[cat_cols]),
    columns=cat_cols,
    index=X_test.index
)

X_train_clean = pd.concat([X_train_num_imputed, X_train_cat_imputed], axis=1)[X_train.columns]
X_test_clean = pd.concat([X_test_num_imputed, X_test_cat_imputed], axis=1)[X_test.columns]

print("\n=== VÉRIFICATION APPRÈS IMPUTATION ===")
print(f"Valeurs manquantes restant dans X_train : {X_train_clean.isnull().sum().sum()}")
print(f"Valeurs manquantes restant dans X_test  : {X_test_clean.isnull().sum().sum()}")

Colonnes numériques (9) : ['Year_of_Study', 'Sleep_Hours', 'Study_Hours_Per_Week', 'GPA', 'Social_Media_Hours', 'Screen_Time_Hours', 'Financial_Stress', 'Family_Support', 'Attendance_Rate']
Colonnes catégorielles (4) : ['Physical_Activity', 'Part_Time_Job', 'Diet_Quality', 'Previous_Mental_Health_History']

=== VÉRIFICATION APPRÈS IMPUTATION ===
Valeurs manquantes restant dans X_train : 0
Valeurs manquantes restant dans X_test  : 0


C:\Users\achra\AppData\Local\Temp\ipykernel_968\4081596708.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()


## Outlier Handling (IQR Capping) & Feature Encoding

In [5]:
X_train_capped = X_train_clean.copy()
X_test_capped = X_test_clean.copy()

outlier_summary = {}

for col in num_cols:
    Q1 = X_train_clean[col].quantile(0.25)
    Q3 = X_train_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    n_outliers = ((X_train_clean[col] < lower_bound) | (X_train_clean[col] > upper_bound)).sum()
    outlier_summary[col] = n_outliers
    
    X_train_capped[col] = np.clip(X_train_clean[col], lower_bound, upper_bound)
    X_test_capped[col] = np.clip(X_test_clean[col], lower_bound, upper_bound)

print("=== OUTLIERS DETECTED & CAPPED PER NUMERICAL FEATURE ===")
print(pd.Series(outlier_summary, name="Train Outliers Count"))

=== OUTLIERS DETECTED & CAPPED PER NUMERICAL FEATURE ===
Year_of_Study            0
Sleep_Hours              9
Study_Hours_Per_Week     7
GPA                      8
Social_Media_Hours      16
Screen_Time_Hours        7
Financial_Stress         0
Family_Support           0
Attendance_Rate         10
Name: Train Outliers Count, dtype: int64


## Standardization of Numerical Features & Unified Pipeline

In [6]:
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), num_cols),
    ('ord', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(categories=[['Poor', 'Average', 'Good'], ['Low', 'Moderate', 'High']]))
    ]), ['Diet_Quality', 'Physical_Activity']),
    ('bin', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(categories=[['No', 'Yes'], ['No', 'Yes']]))
    ]), ['Part_Time_Job', 'Previous_Mental_Health_History']),
])

pipeline = Pipeline([('preprocessor', preprocessor)])

X_train_pipe = pipeline.fit_transform(X_train_capped)
X_test_pipe = pipeline.transform(X_test_capped)

print("\n=== PIPELINE EXECUTION COMPLETED ===")
print(f"X_train_pipe shape: {X_train_pipe.shape} | X_test_pipe shape: {X_test_pipe.shape}")


=== PIPELINE EXECUTION COMPLETED ===
X_train_pipe shape: (1600, 13) | X_test_pipe shape: (400, 13)


## Saving the preprocessed data

In [7]:
X_train_df = pd.DataFrame(X_train_pipe)
X_test_df = pd.DataFrame(X_test_pipe)


y_train_df = pd.DataFrame(y_train, columns=['Mental_Health_Risk'])
y_test_df = pd.DataFrame(y_test, columns=['Mental_Health_Risk'])


X_train_df.to_csv('../data/processed/X_train_processed.csv', index=False)
X_test_df.to_csv('../data/processed/X_test_processed.csv', index=False)
y_train_df.to_csv('../data/processed/y_train.csv', index=False)
y_test_df.to_csv('../data/processed/y_test.csv', index=False)

print("Processed data saved successfully")

Processed data saved successfully